<a href="https://colab.research.google.com/github/liorock1/Partial-SAM-COMP5329/blob/main/PartialSAM_temporal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Partial-SAM — COMP5329 Assignment 2
** Vaibhavi Shivanna · Xiaohe Bu· Lior Sabbagh **

---

## How to use this notebook

| Who | Runs | Section |
|-----|------|---------|
| **Everyone** | Setup (once) | Sections 1–7 |
| **Vaibhavi** | Temporal ablation | Section 8 |
| **Xiaohe** | Structural ablation | Section 9 |
| **Lior** | Joint ablation | Section 10 |
| **Everyone** | Download results | Section 11 |


## Section 1 · GPU Check

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('NO GPU FOUND — go to Runtime > Change runtime type > T4 GPU')

PyTorch: 2.10.0+cu128
GPU available: True
GPU: Tesla T4


## Section 2 · Imports

In [ ]:
import os, csv, time, copy
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models

try:
    import pandas as pd
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    import numpy as np
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'pandas', 'matplotlib'])
    import pandas as pd
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    import numpy as np

print('All imports OK')

All imports OK


## Section 3 · Shared Config

**Do not change these values** — all three experiments must use the same settings.

In [ ]:
# ── Shared hyperparameters (do not change) ───────────────────────────
EPOCHS       = 50
BATCH_SIZE   = 128
LR           = 0.1
MOMENTUM     = 0.9
WEIGHT_DECAY = 5e-4
RHO          = 0.05   # SAM perturbation radius (Foret et al. 2021)
SEEDS        = [42]
NUM_CLASSES  = 100

# CIFAR-100 normalisation (Krizhevsky 2009)
CIFAR_MEAN = (0.5071, 0.4865, 0.4409)
CIFAR_STD  = (0.2673, 0.2564, 0.2761)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTDIR = 'results'
os.makedirs(OUTDIR, exist_ok=True)

print(f'Device : {DEVICE}')
print(f'Epochs : {EPOCHS}')
print(f'Seeds  : {SEEDS}')
print(f'Output : {OUTDIR}/')

Device : cuda
Epochs : 50
Seeds  : [42]
Output : results/


## Section 4 · Dataset — CIFAR-100

Source: Krizhevsky (2009), https://www.cs.toronto.edu/~kriz/cifar.html

Downloads automatically (~160 MB, first run only).

In [ ]:
def get_dataloaders(batch_size=BATCH_SIZE, num_workers=2):
    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    train_set = torchvision.datasets.CIFAR100('./data', train=True,  download=True, transform=train_tf)
    test_set  = torchvision.datasets.CIFAR100('./data', train=False, download=True, transform=test_tf)
    train_loader = torch.utils.data.DataLoader(
        train_set, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=True)
    test_loader = torch.utils.data.DataLoader(
        test_set, batch_size=256, shuffle=False,
        num_workers=num_workers, pin_memory=True)
    return train_loader, test_loader

# Download and verify
train_loader, test_loader = get_dataloaders()
print(f'Train : {len(train_loader.dataset):,} images')
print(f'Test  : {len(test_loader.dataset):,} images')
print(f'Classes: {len(train_loader.dataset.classes)}')

100%|██████████| 169M/169M [00:14<00:00, 11.7MB/s]


Train : 50,000 images
Test  : 10,000 images
Classes: 100


## Section 5 · Model — ResNet-18 for CIFAR-100

Standard CIFAR adaptation (He et al. 2016):
- 3×3 stem conv stride 1 (not 7×7 stride 2)
- No initial max-pool
- Skip connection: H(x) = F(x) + x  [Week 5 tutorial]

In [ ]:
def build_model(seed=42):
    """Fresh ResNet-18 for CIFAR-100. Call once per experiment run."""
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    model = models.resnet18(weights=None, num_classes=NUM_CLASSES)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model.to(DEVICE)

def get_layer_params(model, config):
    """
    Return parameters for specific ResNet-18 residual stages.
    Used by Xiaohe (structural) and Lior (joint).

    config:
        'all4'   -> B1+B2+B3+B4  (full SAM baseline)
        'last2'  -> B3+B4
        'last1'  -> B4 only
        'first1' -> B1 only
        'first2' -> B1+B2
    """
    layer_map = {
        'first1': [model.layer1],
        'first2': [model.layer1, model.layer2],
        'last1':  [model.layer4],
        'last2':  [model.layer3, model.layer4],
        'all4':   [model.layer1, model.layer2, model.layer3, model.layer4],
    }
    return [p for stage in layer_map[config] for p in stage.parameters()]

# Verify
_m = build_model()
n_total = sum(p.numel() for p in _m.parameters())
print(f'ResNet-18 total params: {n_total:,}')
print(f'Output shape: {_m(torch.randn(2,3,32,32).to(DEVICE)).shape}')
print()
print(f'{"Config":<10} {"Perturbed":>12} {"Total":>12} {"% of model":>12}')
print('-' * 50)
for cfg in ['first1','first2','last1','last2','all4']:
    n = sum(p.numel() for p in get_layer_params(_m, cfg))
    print(f'{cfg:<10} {n:>12,} {n_total:>12,} {100*n/n_total:>10.1f}%')
del _m

ResNet-18 total params: 11,220,132
Output shape: torch.Size([2, 100])

Config        Perturbed        Total   % of model
--------------------------------------------------
first1          147,968   11,220,132        1.3%
first2          673,536   11,220,132        6.0%
last1         8,393,728   11,220,132       74.8%
last2        10,493,440   11,220,132       93.5%
all4         11,166,976   11,220,132       99.5%


## Section 6 · Optimizers

- **SAM**: full perturbation over all params (Foret et al. 2021)
- **PartialSAM**: ascent perturbs selected layers only; descent updates all params

In [ ]:
class SAM(optim.Optimizer):
    """Full SAM — perturbs all parameters."""
    def __init__(self, params, base_optimizer, rho=0.05, **kwargs):
        defaults = dict(rho=rho, **kwargs)
        super().__init__(params, defaults)
        self.base_optimizer = base_optimizer(self.param_groups, **kwargs)
        self.param_groups   = self.base_optimizer.param_groups

    @torch.no_grad()
    def first_step(self, zero_grad=False):
        """Ascent: w = w + eps_hat"""
        gn = self._grad_norm()
        for group in self.param_groups:
            scale = group['rho'] / (gn + 1e-12)
            for p in group['params']:
                if p.grad is None: continue
                e_w = p.grad * scale.to(p)
                p.add_(e_w)
                self.state[p]['e_w'] = e_w
        if zero_grad: self.zero_grad()

    @torch.no_grad()
    def second_step(self, zero_grad=False):
        """Descent: undo perturbation, SGD step using gradient at w+eps_hat"""
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None: continue
                p.sub_(self.state[p]['e_w'])
        self.base_optimizer.step()
        if zero_grad: self.zero_grad()

    @torch.no_grad()
    def _grad_norm(self):
        dev = self.param_groups[0]['params'][0].device
        return torch.norm(torch.stack([
            p.grad.norm(p=2).to(dev)
            for group in self.param_groups
            for p in group['params'] if p.grad is not None
        ]), p=2)

    def load_state_dict(self, d):
        super().load_state_dict(d)
        self.base_optimizer.param_groups = self.param_groups


class PartialSAM:
    """Structural Partial-SAM — ascent perturbs selected layers only."""
    def __init__(self, all_params, sam_params, base_cls, rho=0.05, **kwargs):
        self.base_optimizer = base_cls(all_params, **kwargs)
        self.sam_params     = list(sam_params)
        self.rho            = rho

    def zero_grad(self): self.base_optimizer.zero_grad()

    @torch.no_grad()
    def first_step(self):
        grads = [p.grad for p in self.sam_params if p.grad is not None]
        if not grads: return
        gn    = torch.norm(torch.stack([g.norm(p=2) for g in grads]), p=2)
        scale = self.rho / (gn + 1e-12)
        for p in self.sam_params:
            if p.grad is None: continue
            e_w = p.grad * scale
            p.add_(e_w); p._e_w = e_w

    @torch.no_grad()
    def second_step(self):
        for p in self.sam_params:
            if hasattr(p, '_e_w'): p.sub_(p._e_w)
        self.base_optimizer.step()

    def state_dict(self):         return self.base_optimizer.state_dict()
    def load_state_dict(self, d): self.base_optimizer.load_state_dict(d)


print('SAM and PartialSAM defined')

SAM and PartialSAM defined


## Section 7 · Shared Training Utilities

In [ ]:
def train_sgd(model, loader, optimizer, criterion):
    """Standard SGD — one forward-backward pass per step."""
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        out  = model(x); loss = criterion(out, y)
        loss.backward(); optimizer.step()
        total_loss += loss.item() * y.size(0)
        correct    += out.argmax(1).eq(y).sum().item()
        total      += y.size(0)
    return total_loss / total, correct / total


def train_sam(model, loader, optimizer, criterion, is_partial=False):
    """Two-pass SAM update. Works for both SAM and PartialSAM."""
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        # Pass 1: ascent — find eps_hat
        optimizer.zero_grad()
        out = model(x); loss = criterion(out, y); loss.backward()
        if is_partial: optimizer.first_step(); optimizer.zero_grad()
        else:          optimizer.first_step(zero_grad=True)
        # Pass 2: descent — update from w + eps_hat
        out2 = model(x); loss2 = criterion(out2, y); loss2.backward()
        if is_partial: optimizer.second_step()
        else:          optimizer.second_step(zero_grad=True)
        total_loss += loss2.item() * y.size(0)
        correct    += out2.argmax(1).eq(y).sum().item()
        total      += y.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y  = x.to(DEVICE), y.to(DEVICE)
        out   = model(x); loss = criterion(out, y)
        total_loss += loss.item() * y.size(0)
        correct    += out.argmax(1).eq(y).sum().item()
        total      += y.size(0)
    return total_loss / total, correct / total


def measure_sharpness(model, loader, criterion, rho=RHO, n_batches=20):
    """
    rho-sharpness = max_{||eps||<=rho} [L(w+eps) - L(w)]
    Lower = flatter minimum = better generalisation.
    Fixed: separates base loss (no_grad) from gradient computation
    to avoid computation graph errors.
    """
    model.eval()
    base_l, pert_l = [], []
    for i, (x, y) in enumerate(loader):
        if i >= n_batches: break
        x, y = x.to(DEVICE), y.to(DEVICE)

        # Base loss — no grad needed
        with torch.no_grad():
            base_l.append(criterion(model(x), y).item())

        # Fresh forward pass to get gradients for perturbation direction
        model.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()

        # Save weights, apply perturbation
        saved = {n: p.data.clone()
                 for n, p in model.named_parameters() if p.grad is not None}
        gn    = torch.norm(torch.stack(
                    [p.grad.norm() for p in model.parameters()
                     if p.grad is not None]))
        scale = rho / (gn + 1e-12)
        for p in model.parameters():
            if p.grad is not None:
                p.data.add_(p.grad * scale)

        # Perturbed loss — no grad needed
        with torch.no_grad():
            pert_l.append(criterion(model(x), y).item())

        # Restore original weights
        for n, p in model.named_parameters():
            if n in saved: p.data.copy_(saved[n])
        model.zero_grad()

    return max(0.0, sum(pert_l)/len(pert_l) - sum(base_l)/len(base_l))

def save_row(row, filename):
    """Append one result row to a CSV file."""
    path   = os.path.join(OUTDIR, filename)
    exists = os.path.isfile(path)
    with open(path, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=row.keys())
        if not exists: w.writeheader()
        w.writerow(row)


def save_history(history, name):
    """Save per-epoch training history to CSV."""
    path = os.path.join(OUTDIR, f'history_{name}.csv')
    with open(path, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=history[0].keys())
        w.writeheader(); w.writerows(history)


print('All shared utilities defined')
print('Setup complete — scroll to your section and start running')

All shared utilities defined
Setup complete — scroll to your section and start running


---
# SECTION 8 — Temporal Ablation
**Vary when SAM activates. Keep all4 layers throughout.**


In [ ]:
def run_temporal(alpha, seed):
    """
    Temporal Partial-SAM experiment.
    optimizer(t) = SAM if t >= EPOCHS*(1-alpha), else SGD
    """
    sam_start = int(EPOCHS * (1.0 - alpha))
    label     = f'alpha{alpha}_s{seed}'
    criterion = nn.CrossEntropyLoss()

    print(f"\n{'='*60}")
    print(f"TEMPORAL | alpha={alpha} | seed={seed} | SAM from epoch {sam_start}")
    print(f"{'='*60}")

    model   = build_model(seed)
    sgd_opt = optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
    sam_opt = SAM(model.parameters(), optim.SGD, rho=RHO,
                  lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY) if alpha > 0 else None
    scheduler = optim.lr_scheduler.CosineAnnealingLR(sgd_opt, T_max=EPOCHS)

    history = []; best_acc = 0.0; best_state = None

    for epoch in range(1, EPOCHS + 1):
        t0      = time.time()
        use_sam = (alpha > 0.0) and (epoch > sam_start)
        if use_sam:
            tr_loss, tr_acc = train_sam(model, train_loader, sam_opt, criterion)
        else:
            tr_loss, tr_acc = train_sgd(model, train_loader, sgd_opt, criterion)
        _, te_acc = evaluate(model, test_loader, criterion)
        scheduler.step()

        history.append({'epoch': epoch, 'phase': 'SAM' if use_sam else 'SGD',
                         'train_acc': round(tr_acc*100,2), 'test_acc': round(te_acc*100,2)})
        if te_acc > best_acc: best_acc = te_acc; best_state = copy.deepcopy(model.state_dict())
        if epoch % 40 == 0 or epoch == 1 or epoch == sam_start + 1:
            tag = '[SAM]' if use_sam else '[SGD]'
            print(f"  Epoch {epoch:3d}/{EPOCHS} {tag}  train {tr_acc*100:.1f}%  test {te_acc*100:.1f}%  ({time.time()-t0:.0f}s)")

    model.load_state_dict(best_state)
    _, final_acc = evaluate(model, test_loader, criterion)
    print('  Measuring sharpness...')
    sharpness = measure_sharpness(model, train_loader, criterion)

    sam_eps   = EPOCHS - sam_start
    flops_saved = round((1 - (sam_start + sam_eps*2) / (EPOCHS*2)) * 100, 1)

    row = {'alpha': alpha, 'seed': seed,
           'test_acc': round(final_acc*100,2),
           'sharpness': round(sharpness,4),
           'flops_saved': flops_saved}
    save_row(row, 'results_temporal.csv')
    save_history(history, label)
    print(f"  Done — acc={final_acc*100:.2f}%  sharp={sharpness:.4f}  flops_saved={flops_saved}%")
    return row

print('run_temporal() defined')

run_temporal() defined


In [ ]:

# Runs all 15 temporal experiments back to back. ~6 hrs on Colab T4.
# Auto-skips completed runs if session resets — safe to rerun.

ALPHAS = [0.0, 0.25, 1.00]

# Load completed runs
done_t = set()
t_csv  = os.path.join(OUTDIR, 'results_temporal.csv')
if os.path.exists(t_csv):
    df_done = pd.read_csv(t_csv)
    for _, r in df_done.iterrows():
        done_t.add((float(r['alpha']), int(r['seed'])))
    print(f'Already done: {len(done_t)} runs — skipping them')

remaining = [(a, s) for a in ALPHAS for s in SEEDS if (a,s) not in done_t]
print(f'{len(remaining)}/{len(ALPHAS)*len(SEEDS)} runs remaining (~{len(remaining)*25} min)\n')

for alpha, seed in remaining:
    run_temporal(alpha, seed)

print('\nAll temporal runs complete!')

3/3 runs remaining (~75 min)


TEMPORAL | alpha=0.0 | seed=42 | SAM from epoch 50
  Epoch   1/50 [SGD]  train 9.1%  test 14.3%  (41s)
  Epoch  40/50 [SGD]  train 98.5%  test 74.0%  (44s)
  Measuring sharpness...
  Done — acc=75.71%  sharp=0.0929  flops_saved=50.0%

TEMPORAL | alpha=0.25 | seed=42 | SAM from epoch 37
  Epoch   1/50 [SGD]  train 9.4%  test 14.8%  (45s)
  Epoch  38/50 [SAM]  train 55.2%  test 54.2%  (87s)
  Epoch  40/50 [SAM]  train 66.0%  test 60.2%  (87s)
  Measuring sharpness...
  Done — acc=71.85%  sharp=0.2083  flops_saved=37.0%

TEMPORAL | alpha=1.0 | seed=42 | SAM from epoch 0
  Epoch   1/50 [SAM]  train 8.2%  test 14.1%  (87s)


/tmp/ipykernel_701/2863185570.py:30: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


  Epoch  40/50 [SAM]  train 68.5%  test 63.0%  (87s)
  Measuring sharpness...
  Done — acc=64.10%  sharp=0.2033  flops_saved=0.0%

All temporal runs complete!
